# Mirage cloak lab (Stable Diffusion 1.5 VAE + img2img simulator)

Run this notebook in a Colab **GPU (T4)** runtime from the Mirage checkout so it can use `backend/assets/target.png`. If the target asset is not found, the notebook asks you to upload that file. It then asks for one photo, runs the **standard** PGD preset from `02_backend_prd.md` section 5, and displays the original, quantized PNG cloak, and an epsilon-normalized difference heatmap. The final cell runs all four section 7 img2img presets on both images with matching settings and seed, then displays each pair.

The face mask uses the agreed 12 px Gaussian-blur sigma. The latent score below checks the final quantized PNG against the fixed target latent; it is a heuristic, not a substitute for the img2img gate test. Use only a photo you have permission to process.

Run the setup cell first, then restart the Colab runtime before running the remaining cells. Setup removes the optional `torchao` package after installing the notebook dependencies because an incompatible `torchao` API can break Diffusers imports. A warning that Triton is unavailable is separate and usually harmless for this float32 VAE workflow.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors pillow numpy opencv-python-headless mediapipe scikit-image
!pip -q uninstall -y torchao

In [ ]:
from pathlib import Path
import math
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFilter, ImageOps
from tqdm.auto import tqdm
from diffusers import AutoencoderKL
from google.colab import files

SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
STRENGTH = "standard"
PRESETS = {"light": (6, 40), "standard": (12, 80), "strong": (16, 120)}
FACE_FOCUS = True
ROBUST_MODE = True
FACE_MASK_BLUR_SIGMA_PX = 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cpu":
    print("CPU mode: standard preset uses 0.4 x the GPU step count.")

vae = AutoencoderKL.from_pretrained(
    SD_REPO, subfolder="vae", torch_dtype=torch.float32
).to(device).eval()
for parameter in vae.parameters():
    parameter.requires_grad_(False)
print(f"Loaded float32 VAE from {SD_REPO}.")

In [ ]:
target_candidates = [
    Path.cwd() / "backend" / "assets" / "target.png",
    Path.cwd() / "assets" / "target.png",
    Path.cwd().parent / "backend" / "assets" / "target.png",
    Path.cwd().parent / "assets" / "target.png",
]
target_path = next((path for path in target_candidates if path.is_file()), None)
if target_path is None:
    print("Upload the repository's backend/assets/target.png fixed target pattern.")
    uploaded_target = files.upload()
    if "target.png" not in uploaded_target:
        raise FileNotFoundError("Expected the fixed target asset named target.png.")
    target_path = Path("target.png")
target_image = Image.open(target_path).convert("RGB")
print(f"Using fixed target: {target_path}")

print("Upload one photo to cloak.")
uploaded_photo = files.upload()
if not uploaded_photo:
    raise ValueError("No photo was uploaded.")
photo_name, photo_bytes = next(iter(uploaded_photo.items()))
from io import BytesIO

original_image = ImageOps.exif_transpose(Image.open(BytesIO(photo_bytes))).convert("RGB")
max_side = 768 if device.type == "cuda" else 512
scale = min(1.0, max_side / max(original_image.size))
width = max(8, (int(original_image.width * scale) // 8) * 8)
height = max(8, (int(original_image.height * scale) // 8) * 8)
original_image = original_image.resize((width, height), Image.Resampling.LANCZOS)
working_size = (width, height)

def image_to_tensor(image):
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(device)

x = image_to_tensor(original_image)
target_image = target_image.resize(working_size, Image.Resampling.BICUBIC)
target = image_to_tensor(target_image)
with torch.no_grad():
    z_target = vae.encode(target * 2 - 1).latent_dist.mean
    z_original = vae.encode(x * 2 - 1).latent_dist.mean
print(f"Photo: {photo_name}; working size: {width} x {height}")

In [ ]:
def gaussian_blur_tensor(image, sigma):
    if sigma <= 1e-6:
        return image
    radius = max(1, math.ceil(3 * sigma))
    coordinates = torch.arange(-radius, radius + 1, device=image.device, dtype=image.dtype)
    kernel_1d = torch.exp(-(coordinates ** 2) / (2 * sigma ** 2))
    kernel_1d = kernel_1d / kernel_1d.sum()
    kernel_2d = kernel_1d[:, None] * kernel_1d[None, :]
    channels = image.shape[1]
    kernel = kernel_2d.expand(channels, 1, -1, -1).contiguous()
    padded = F.pad(image, (radius, radius, radius, radius), mode="reflect")
    return F.conv2d(padded, kernel, groups=channels)


def augment(image):
    """Differentiable EOT augmentations specified in PRD section 5.4."""
    sigma_blur = random.uniform(0.0, 1.0)
    augmented = gaussian_blur_tensor(image, sigma_blur)
    sigma_noise = random.uniform(0.0, 0.02)
    if sigma_noise > 0:
        augmented = augmented + torch.randn_like(augmented) * sigma_noise
    augmented = augmented.clamp(0, 1)
    height, width = augmented.shape[-2:]
    scale = random.uniform(0.75, 1.0)
    if scale < 1.0:
        small_size = (max(1, round(height * scale)), max(1, round(width * scale)))
        augmented = F.interpolate(
            augmented, size=small_size, mode="bilinear", align_corners=False
        )
        augmented = F.interpolate(
            augmented, size=(height, width), mode="bilinear", align_corners=False
        )
    return augmented


def detect_face_boxes(image):
    rgb = np.asarray(image.convert("RGB"))
    try:
        import mediapipe as mp
        face_detection_api = getattr(getattr(mp, "solutions", None), "face_detection", None)
        if face_detection_api is None:
            from mediapipe.python.solutions import face_detection as face_detection_api
    except (ImportError, AttributeError):
        face_detection_api = None

    boxes = []
    if face_detection_api is not None:
        with face_detection_api.FaceDetection(
            model_selection=1, min_detection_confidence=0.5
        ) as detector:
            result = detector.process(rgb)
        for detection in result.detections or []:
            box = detection.location_data.relative_bounding_box
            left = round(box.xmin * image.width)
            top = round(box.ymin * image.height)
            right = round((box.xmin + box.width) * image.width)
            bottom = round((box.ymin + box.height) * image.height)
            boxes.append((left, top, right, bottom))
    else:
        print("MediaPipe FaceDetection API unavailable; using OpenCV Haar fallback.")
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        detector = cv2.CascadeClassifier(cascade_path)
        if detector.empty():
            raise RuntimeError("OpenCV Haar face detector could not be loaded.")
        for x0, y0, box_width, box_height in detector.detectMultiScale(gray):
            boxes.append((x0, y0, x0 + box_width, y0 + box_height))
    return boxes


def build_face_mask(image, face_focus=True):
    width, height = image.size
    full_mask = torch.ones((1, 1, height, width), device=device, dtype=torch.float32)
    if not face_focus:
        return full_mask, False

    boxes = detect_face_boxes(image)
    if not boxes:
        return full_mask, False

    face_image = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(face_image)
    for left, top, right, bottom in boxes:
        center_x = (left + right) / 2
        center_y = (top + bottom) / 2
        expanded_width = (right - left) * 1.8
        expanded_height = (bottom - top) * 1.8
        expanded_box = (
            max(0, round(center_x - expanded_width / 2)),
            max(0, round(center_y - expanded_height / 2)),
            min(width, round(center_x + expanded_width / 2)),
            min(height, round(center_y + expanded_height / 2)),
        )
        draw.rectangle(expanded_box, fill=255)
    face_image = face_image.filter(
        ImageFilter.GaussianBlur(radius=FACE_MASK_BLUR_SIGMA_PX)
    )
    face_mask = torch.from_numpy(np.asarray(face_image, dtype=np.float32) / 255.0)
    face_mask = face_mask.to(device=device, dtype=torch.float32)[None, None]
    return 0.4 + 0.6 * face_mask, True


mask, face_found = build_face_mask(original_image, face_focus=FACE_FOCUS)
print(f"Face focus: {FACE_FOCUS}; face found: {face_found}; robust mode: {ROBUST_MODE}")

In [ ]:
def cloak_pgd(x, vae, z_target, eps, steps, mask, cb, robust=False):
    alpha = 2.5 * eps / steps
    delta = ((torch.rand_like(x) * 2 - 1) * eps * 0.1) * mask
    for i in range(steps):
        delta.requires_grad_(True)
        x_adv = torch.clamp(x + delta, 0, 1)
        inp = augment(x_adv) if robust else x_adv
        z = vae.encode(inp * 2 - 1).latent_dist.mean
        loss = F.mse_loss(z, z_target)
        (grad,) = torch.autograd.grad(loss, delta)
        delta = (delta - alpha * grad.sign()).detach()
        delta = torch.max(torch.min(delta, eps * mask), -eps * mask)
        if i % 4 == 0:
            cb(i / steps)
    return torch.clamp(x + delta, 0, 1).detach()


eps_255, gpu_steps = PRESETS[STRENGTH]
eps = eps_255 / 255.0
steps = int(gpu_steps * (0.4 if device.type == "cpu" else 1.0))
progress = tqdm(total=steps, desc=f"PGD {STRENGTH}")

def update_progress(fraction):
    desired = min(steps, round(fraction * steps) + 1)
    progress.update(max(0, desired - progress.n))

cloaked_float = cloak_pgd(
    x=x,
    vae=vae,
    z_target=z_target,
    eps=eps,
    steps=steps,
    mask=mask,
    cb=update_progress,
    robust=ROBUST_MODE,
)
progress.update(steps - progress.n)
progress.close()
print(f"Completed {steps} steps at eps={eps_255}/255.")

In [ ]:
cloaked_array = (
    cloaked_float[0].permute(1, 2, 0).cpu().numpy().clip(0, 1) * 255
).round().astype(np.uint8)
original_array = np.asarray(original_image, dtype=np.uint8)
cloaked_image = Image.fromarray(cloaked_array, mode="RGB")
OUTPUT_PATH = Path("cloaked_standard.png")
cloaked_image.save(OUTPUT_PATH, format="PNG")

with torch.no_grad():
    x_quantized = image_to_tensor(cloaked_image)
    z_quantized = vae.encode(x_quantized * 2 - 1).latent_dist.mean

original_distance = torch.linalg.vector_norm(z_original - z_target)
quantized_distance = torch.linalg.vector_norm(z_quantized - z_target)
if original_distance.item() == 0:
    raise RuntimeError("Original image latent equals the target latent; score is undefined.")
protection_score = torch.clamp(
    100 * (1 - quantized_distance / original_distance), min=0, max=100
).item()
print(f"Saved final uint8 PNG: {OUTPUT_PATH.resolve()}")
print(f"Post-quantization protection score (heuristic): {protection_score:.1f}/100")

difference = np.abs(cloaked_array.astype(np.float32) - original_array.astype(np.float32)).mean(axis=2)
difference_normalized = np.clip(difference / eps_255, 0, 1)
difference_color = cv2.applyColorMap(
    np.round(difference_normalized * 255).astype(np.uint8), cv2.COLORMAP_INFERNO
)
difference_color = cv2.cvtColor(difference_color, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(original_array)
axes[0].set_title("Original")
axes[1].imshow(cloaked_array)
axes[1].set_title("Cloaked (standard)")
axes[2].imshow(difference_color)
axes[2].set_title("Mean absolute difference (normalized by eps)")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

files.download(str(OUTPUT_PATH))

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
from skimage.metrics import structural_similarity

SIM_PRESETS = {
    "background_snow": "a portrait photo of the same person standing in a snowy mountain landscape, realistic",
    "background_beach": "a portrait photo of the same person on a sunny beach, realistic",
    "new_hairstyle": "a portrait photo of the same person with a different hairstyle, realistic",
    "vintage_style": "a vintage film photograph of the same person, 1970s style",
}
SIM_SEED = 1234
SIM_STRENGTH = 0.6
SIM_GUIDANCE_SCALE = 7.5
SIM_STEPS = 25

if device.type != "cuda":
    raise RuntimeError("The section 7 simulator requires a GPU; select a Colab T4 runtime.")

# The cloak optimization is complete. Free its float32 VAE before loading img2img.
if "vae" in globals():
    del vae
torch.cuda.empty_cache()
simulator = StableDiffusionImg2ImgPipeline.from_pretrained(
    SD_REPO, torch_dtype=torch.float16
).to(device)
simulator.set_progress_bar_config(disable=True)

def prepare_simulator_image(image):
    scale = min(1.0, 512 / max(image.size))
    sim_width = max(8, (int(image.width * scale) // 8) * 8)
    sim_height = max(8, (int(image.height * scale) // 8) * 8)
    return image.resize((sim_width, sim_height), Image.Resampling.LANCZOS)

sim_original = prepare_simulator_image(original_image)
sim_cloaked = prepare_simulator_image(cloaked_image)
if sim_original.size != sim_cloaked.size:
    raise ValueError("Simulator original and cloaked inputs must have identical dimensions.")

simulator_results = {}
sim_output_dir = Path("simulator_outputs")
sim_output_dir.mkdir(exist_ok=True)
for preset_id, prompt in tqdm(SIM_PRESETS.items(), desc="Img2img presets"):
    edited_pair = []
    for input_image in (sim_original, sim_cloaked):
        generator = torch.Generator(device=device).manual_seed(SIM_SEED)
        result = simulator(
            prompt=prompt,
            image=input_image,
            strength=SIM_STRENGTH,
            guidance_scale=SIM_GUIDANCE_SCALE,
            num_inference_steps=SIM_STEPS,
            generator=generator,
        )
        edited_image = result.images[0]
        safety_flags = getattr(result, "nsfw_content_detected", None)
        if safety_flags is not None and len(safety_flags) > 0 and safety_flags[0]:
            raise RuntimeError(f"SIM_BLOCKED: safety checker blocked {preset_id}.")
        if np.asarray(edited_image.convert("RGB")).max() == 0:
            raise RuntimeError(f"SIM_BLOCKED: simulator returned a black image for {preset_id}.")
        edited_pair.append(edited_image.convert("RGB"))
        del result, generator
        if device.type == "cuda":
            torch.cuda.empty_cache()

    edited_original, edited_cloaked = edited_pair
    edited_original_array = np.asarray(edited_original, dtype=np.uint8)
    edited_cloaked_array = np.asarray(edited_cloaked, dtype=np.uint8)
    similarity = structural_similarity(
        edited_original_array,
        edited_cloaked_array,
        channel_axis=2,
        data_range=255,
    )
    disruption_score = float(np.clip(100 * (1 - similarity), 0, 100))
    simulator_results[preset_id] = {
        "prompt": prompt,
        "edit_on_original": edited_original,
        "edit_on_cloaked": edited_cloaked,
        "seed": SIM_SEED,
        "disruption_score_heuristic": disruption_score,
    }
    edited_original.save(sim_output_dir / f"sim_{preset_id}_original.png")
    edited_cloaked.save(sim_output_dir / f"sim_{preset_id}_cloaked.png")
    print(f"{preset_id}: disruption score (heuristic) = {disruption_score:.1f}/100")

del simulator
torch.cuda.empty_cache()

for preset_id, result in simulator_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(result["edit_on_original"])
    axes[0].set_title("Edit from original")
    axes[1].imshow(result["edit_on_cloaked"])
    axes[1].set_title("Edit from cloaked")
    fig.suptitle(
        f"{preset_id} | same prompt and seed {result['seed']} | "
        f"disruption heuristic {result['disruption_score_heuristic']:.1f}/100"
    )
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

print(f"Simulator PNG pairs saved in: {sim_output_dir.resolve()}")